# DMBI Experiment 6 – Implementation of K-Means Clustering Using Python

**Name:** Sahil Setya  
**Class:** D15A  
**Roll No.:** 48  
**Subject:** DMBI

**Dataset required:** `Music_Streaming_Subscriber_Churn.csv` (500 records, 10 columns). The practical document does not contain the complete CSV dataset. Upload the original CSV to Colab before running the dataset-loading cell.

In [ ]:
# 1. Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

print('All libraries imported successfully!')

In [ ]:
# 2. Upload and load the dataset in Google Colab
from google.colab import files
uploaded = files.upload()

file_name = 'Music_Streaming_Subscriber_Churn.csv'
if file_name not in uploaded:
    raise FileNotFoundError(f'Please upload {file_name}')

df = pd.read_csv(file_name)
print('Dataset loaded successfully!')
print('Dataset Shape:', df.shape)
print('\nFirst 5 Rows:')
display(df.head())
print('\nColumn Names:', df.columns.tolist())

In [ ]:
# 3. Select numerical features and preprocess the data
features = ['age', 'listening_hours_per_week', 'monthly_logins', 'tenure_months']
X = df[features].copy()

print('Missing values:')
print(X.isnull().sum())

# Fill missing numerical values with the median
X = X.fillna(X.median())

# Standardize the selected features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('\nData preprocessing completed!')
print('Number of records:', X_scaled.shape[0])
print('Number of features:', X_scaled.shape[1])
print('Selected features:', features)

In [ ]:
# 4. Elbow Method: calculate inertia for K = 1 to 10
inertia_values = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertia_values, marker='o')
plt.title('Elbow Method - Music Streaming Dataset')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(list(k_range))
plt.grid(True)
plt.show()

In [ ]:
# 5. Apply K-Means clustering with K = 3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df['Cluster'] = clusters

print('K-Means clustering completed!')
print('Number of clusters:', len(set(clusters)))
print('\nCluster Distribution:')
print(df['Cluster'].value_counts().sort_index())
print('\nCluster Centroids (standardized features):')
print(kmeans.cluster_centers_)

In [ ]:
# 6. Evaluate clustering performance
sil_score = silhouette_score(X_scaled, clusters)
inertia = kmeans.inertia_
db_index = davies_bouldin_score(X_scaled, clusters)

print('K-MEANS CLUSTERING EVALUATION')
print('============================')
print('Number of Clusters:', len(set(clusters)))
print('Silhouette Score:', round(sil_score, 3))
print('Inertia:', round(inertia, 4))
print('Davies-Bouldin Index:', round(db_index, 4))

In [ ]:
# 7. Visualize clusters using standardized listening hours and monthly logins
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=X_scaled[:, 1],
    y=X_scaled[:, 2],
    hue=clusters,
    palette='viridis',
    s=60
)
plt.title('K-Means Clustering - Music Streaming Users')
plt.xlabel('Standardized Listening Hours per Week')
plt.ylabel('Standardized Monthly Logins')
plt.legend(title='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 8. Cluster-wise analysis: feature means and number of users
cluster_analysis = df.groupby('Cluster').agg({
    'age': 'mean',
    'listening_hours_per_week': 'mean',
    'monthly_logins': 'mean',
    'tenure_months': 'mean',
    'user_id': 'count'
}).round(2)
cluster_analysis = cluster_analysis.rename(columns={'user_id': 'Number of Users'})

print('CLUSTER-WISE ANALYSIS')
display(cluster_analysis)

# Plot the number of users in each cluster
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x='Cluster')
plt.title('Number of Users in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Users')
plt.show()

In [ ]:
# 9. Final evaluation summary
print('FINAL K-MEANS EVALUATION SUMMARY')
print('================================')
print('Number of Clusters:', len(set(clusters)))
print('Silhouette Score:', round(sil_score, 3))
print('Inertia:', round(inertia, 4))
print('Davies-Bouldin Index:', round(db_index, 4))
print('\nCluster Distribution:')
print(df['Cluster'].value_counts().sort_index())